# Pipeline in Machine Learning: Full Guide

This notebook covers Pipeline from basic theory to practical usage.

## What you will learn
- What a Pipeline is and why it is useful
- How Pipeline prevents data leakage
- How Pipeline and ColumnTransformer work together
- End-to-end implementation with clean, easy code
- Common mistakes and best practices

## 1. Why Do We Need Pipeline?

In real projects, machine learning is not just model training. We also do preprocessing:
- Missing value handling
- Encoding categorical variables
- Feature scaling

If preprocessing and model training are done separately, mistakes happen easily.

### Problems without pipeline
- Train/test preprocessing mismatch
- Data leakage (using information from test set during training)
- Hard-to-maintain code
- Errors in deployment

`Pipeline` solves this by chaining steps into one reusable workflow.

## 2. Core Idea (Theory)

A pipeline is an ordered sequence of steps:

$$
\text{Raw Data} \rightarrow \text{Preprocessing} \rightarrow \text{Model} \rightarrow \text{Prediction}
$$

Each step except the last should implement:
- `fit()`
- `transform()`

The last step should implement:
- `fit()`
- `predict()` (or `predict_proba()`)

So you call `pipeline.fit(X_train, y_train)` once, and all steps are handled in the correct order.

## 3. Pipeline + ColumnTransformer Together

`ColumnTransformer` handles **different preprocessing for different columns**.

Typical structure:
1. `ColumnTransformer` inside pipeline step `preprocessor`
2. Model as final step (for example, `LogisticRegression`)

This gives one clean object for:
- Training
- Prediction
- Cross-validation
- Saving and deployment

In [ ]:
# 1) Imports
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# 2) Create a small mixed-type dataset
# Includes missing values to show realistic preprocessing
df = pd.DataFrame({
    'age': [25, 32, 47, 51, 62, 23, 40, 36, None, 29],
    'salary': [30000, 50000, 70000, 75000, 90000, 28000, 62000, None, 58000, 45000],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Pune', 'Delhi', 'Mumbai', 'Delhi', None],
    'gender': ['M', 'F', 'F', 'M', 'M', 'F', 'M', 'F', 'F', 'M'],
    'bought': [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]
})

df

In [ ]:
# 3) Separate features and target, then split data
X = df.drop('bought', axis=1)
y = df['bought']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
# 4) Define column groups
numeric_features = ['age', 'salary']
categorical_features = ['city', 'gender']

In [ ]:
# 5) Build preprocessing pipelines for each column type
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine both using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [ ]:
# 6) Create final model pipeline: preprocessor + classifier
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

# Fit once: all preprocessing + model training happens internally
clf.fit(X_train, y_train)

In [ ]:
# 7) Predict and evaluate
y_pred = clf.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# 8) View transformed feature names from the fitted preprocessor
feature_names = clf.named_steps['preprocessor'].get_feature_names_out()

print('Total transformed features:', len(feature_names))
print(feature_names)

In [ ]:
# 9) Cross-validation with the full pipeline (safe and leakage-resistant)
cv_scores = cross_val_score(clf, X, y, cv=3, scoring='accuracy')

print('CV accuracy scores:', cv_scores)
print('Mean CV accuracy:', round(cv_scores.mean(), 3))

## 4. Why Pipeline + ColumnTransformer Makes Tasks Easier

When both are used together:
- You write preprocessing logic once
- Same logic is automatically applied during training and prediction
- Train/test preprocessing mismatch is avoided
- Cross-validation becomes cleaner and safer
- Deployment is easier because everything is wrapped in one object (`clf`)

## 5. Common Mistakes and How to Avoid Them

1. Fitting preprocessors on full dataset before split
   - Correct: split first, then fit on training data only

2. Forgetting `handle_unknown='ignore'` in `OneHotEncoder`
   - New categories in test/production can break prediction without it

3. Scaling categorical one-hot columns unnecessarily
   - Scale numeric columns only

4. Doing manual preprocessing outside pipeline and then using pipeline
   - Keep preprocessing inside pipeline for consistency

5. Not using cross-validation on the full pipeline
   - Use `cross_val_score(clf, X, y, ...)`, not on partially preprocessed data

## 6. Quick Revision Cheatsheet

- `ColumnTransformer`: apply different transforms to different columns
- `Pipeline`: chain preprocessors + model in one object
- `clf.fit(X_train, y_train)`: trains entire workflow
- `clf.predict(X_test)`: applies same preprocessing automatically, then predicts
- `clf.fit_transform(X_test)`: if you are not training model through pipeline
- `clf.named_steps`: inspect inner components

In short:

$$
\text{Column-wise preprocessing} + \text{Model training} = \text{clean and reliable ML workflow}
$$